# ตัวอย่างประกอบการสอน: Graphical vs Simplex Methods

**แนวคิดหลักของสไลด์ (หน้า 54):** *"Simplex เดินจาก 'จุดมุม' หนึ่งไปยังจุดมุมถัดไปที่ทำให้ Objective ดีขึ้น จนไม่มีทางดีขึ้นต่อ"*

ไฟล์นี้เอา**โจทย์เดียวกัน** (Example 1 จากสไลด์ Simplex) มาแก้ **2 วิธีพร้อมกัน** แล้วพิสูจน์ให้เห็นภาพจริงว่า:

> ทุกครั้งที่ Simplex ทำ pivot 1 ครั้ง = มันกำลัง **"กระโดด" จากจุดมุมหนึ่งไปอีกจุดมุมหนึ่งบนกราฟเดียวกับที่ Graphical Method หาเจอ**

**โจทย์:** Maximize $Z=3x_1+5x_2$

ข้อจำกัด: $x_1\le4,\quad 2x_2\le12,\quad 3x_1+2x_2\le18,\quad x_1,x_2\ge0$


In [12]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

pd.set_option('display.precision', 2)


In [ ]:
# 🎨 Helper functions: ใช้ร่วมกันทุกกราฟ เพื่อให้สไตล์สม่ำเสมอและไม่ต้องเขียนซ้ำ
PLOT_RANGE = (-0.5, 7)
FEASIBLE_COLOR = "rgba(100,200,150,0.30)"
CONSTRAINT_COLOR = "steelblue"


def feasible_mask(A, b, xr=PLOT_RANGE, yr=PLOT_RANGE, n=200):
    xs = np.linspace(*xr, n)
    ys = np.linspace(*yr, n)
    X, Y = np.meshgrid(xs, ys)
    feasible = np.ones_like(X, dtype=bool)
    for (a1, a2), bi in zip(A, b):
        feasible &= (a1 * X + a2 * Y <= bi + 1e-9)
    feasible &= (X >= 0) & (Y >= 0)
    return xs, ys, feasible


def add_feasible_region(fig, A, b, xr=PLOT_RANGE, yr=PLOT_RANGE):
    """วาดพื้นที่ feasible region แรเงา คืนค่า xs, ys สำหรับใช้วาดเส้นต่อ"""
    xs, ys, feasible = feasible_mask(A, b, xr, yr)
    fig.add_trace(go.Contour(
        x=xs, y=ys, z=feasible.astype(int), showscale=False,
        contours={"start": 0.5, "end": 1, "size": 1},
        colorscale=[[0, "rgba(0,0,0,0)"], [1, FEASIBLE_COLOR]],
        line={"width": 0}, hoverinfo="skip", name="Feasible Region", showlegend=False,
    ))
    return xs, ys


def add_constraint_lines(fig, A, b, xs, yr=PLOT_RANGE, showlegend=True):
    """วาดเส้นข้อจำกัดทุกเส้น (dashed)"""
    for i, ((a1, a2), bi) in enumerate(zip(A, b)):
        if a2 != 0:
            yy = (bi - a1 * xs) / a2
            fig.add_trace(go.Scatter(x=xs, y=yy, mode="lines",
                                      line={"dash": "dash", "color": CONSTRAINT_COLOR, "width": 1.5},
                                      name=f"เส้นข้อจำกัด {i+1}: {a1:g}x₁+{a2:g}x₂={bi:g}",
                                      showlegend=showlegend))
        else:
            fig.add_trace(go.Scatter(x=[bi / a1] * 2, y=list(yr), mode="lines",
                                      line={"dash": "dash", "color": CONSTRAINT_COLOR, "width": 1.5},
                                      name=f"เส้นข้อจำกัด {i+1}: x₁={bi:g}",
                                      showlegend=showlegend))


def base_figure(A, b, xr=PLOT_RANGE, yr=PLOT_RANGE):
    """สร้างกราฟตั้งต้น: feasible region + เส้นข้อจำกัด + จัด layout มาตรฐาน"""
    fig = go.Figure()
    xs, ys = add_feasible_region(fig, A, b, xr, yr)
    add_constraint_lines(fig, A, b, xs, yr)
    fig.update_layout(
        xaxis_title="x₁", yaxis_title="x₂",
        xaxis={"range": list(xr)}, yaxis={"range": list(yr)},
        width=750, height=600, hovermode="closest", template="plotly_white",
    )
    return fig

## ส่วนที่ 1 — Graphical Method: หาจุดมุมทั้งหมดก่อน (ภาพ "ก่อน" ที่ยังไม่รู้คำตอบ)

วิธีกราฟจะหาจุดมุม (corner point) **ทุกจุด** ของ feasible region แล้วเทียบค่า Z — เป็นการมองภาพรวมทั้งหมดของปัญหา
ก่อนจะรู้ว่าคำตอบคือจุดไหน


In [13]:
c = [3, 5]
A = [[1, 0], [0, 2], [3, 2]]
b = [4, 12, 18]


def find_corners(A, b):
    """หาจุดมุมทั้งหมดของ feasible region จากจุดตัดของทุกคู่เส้น (รวมแกน x=0, y=0)"""
    lines = list(A) + [[1, 0], [0, 1]]
    rhs = list(b) + [0, 0]
    corners = []
    for i in range(len(lines)):
        for j in range(i + 1, len(lines)):
            Amat = np.array([lines[i], lines[j]])
            bvec = np.array([rhs[i], rhs[j]])
            if abs(np.linalg.det(Amat)) < 1e-9:
                continue
            pt = np.linalg.solve(Amat, bvec)
            if pt[0] >= -1e-6 and pt[1] >= -1e-6:
                if all(a1 * pt[0] + a2 * pt[1] <= bi + 1e-6 for (a1, a2), bi in zip(A, b)):
                    corners.append(tuple(np.round(pt, 4)))
    return sorted(set(corners))


corners = find_corners(A, b)
corner_table = pd.DataFrame({
    "จุดมุม": [f"({x:.0f},{y:.0f})" for x, y in corners],
    "Z=3x1+5x2": [c[0] * x + c[1] * y for x, y in corners],
})
corner_table = corner_table.sort_values("Z=3x1+5x2").reset_index(drop=True)
print("=== ตารางจุดมุมทั้งหมด (Graphical Method) ===")
print(corner_table)
best_row = corner_table.loc[corner_table["Z=3x1+5x2"].idxmax()]
print(f"\n>>> Optimal: {best_row['จุดมุม']}  Z={best_row['Z=3x1+5x2']:.0f}")


=== ตารางจุดมุมทั้งหมด (Graphical Method) ===
  จุดมุม  Z=3x1+5x2
0  (0,0)        0.0
1  (4,0)       12.0
2  (4,3)       27.0
3  (0,6)       30.0
4  (2,6)       36.0

>>> Optimal: (2,6)  Z=36


In [16]:
# 📊 Visualization: Graphical Method - จุดมุมบน Feasible Region
fig = base_figure(A, b)

corner_x, corner_y = zip(*corners)
corner_z = [c[0] * x + c[1] * y for x, y in corners]
fig.add_trace(go.Scatter(
    x=corner_x, y=corner_y, mode="markers+text",
    marker={"size": 14, "color": "red", "symbol": "star", "line": {"width": 2, "color": "darkred"}},
    text=[f"({x:.0f},{y:.0f})<br>Z={z:.0f}" for x, y, z in zip(corner_x, corner_y, corner_z)],
    textposition="top center", textfont={"size": 11, "color": "darkred"},
    name="จุดมุม (Corner Points)",
    hovertemplate="<b>จุดมุม: (%{x:.0f}, %{y:.0f})</b><extra></extra>",
))

fig.update_layout(title="<b>📊 Graphical Method: ค้นหาจุดมุมทั้งหมดของ Feasible Region</b>")
fig.show()

print(f"✓ พบจุดมุม {len(corners)} จุด — Graphical Method ต้องเช็คทุกจุดเหล่านี้ก่อนตัดสินใจ")

✓ พบจุดมุม 5 จุด — Graphical Method เช็คทุกจุดเหล่านี้


## ส่วนที่ 2 — Simplex Method: ดู "ก่อน/หลัง" ทีละ pivot

Simplex **ไม่รู้จักจุดมุมทั้งหมดล่วงหน้าเหมือน Graphical** — มันเริ่มจากจุดเดียว (origin) แล้วค่อยๆ **เดินทีละก้าว**
ไปยังจุดมุมที่ดีกว่า จนกว่าจะเจอจุดที่ดีที่สุด เราจะพิมพ์สถานะ **ก่อน pivot** และ **หลัง pivot** ของทุกรอบ


In [14]:
def simplex_maximize(c, A, b, var_names=None):
    c = np.array(c, dtype=float); A = np.array(A, dtype=float); b = np.array(b, dtype=float)
    n_vars = len(c); n_cons = len(b)
    if var_names is None:
        var_names = [f"x{i+1}" for i in range(n_vars)]
    slack_names = [f"S{i+1}" for i in range(n_cons)]
    all_names = var_names + slack_names + ["RHS"]

    tableau = np.zeros((n_cons + 1, n_vars + n_cons + 1))
    tableau[:n_cons, :n_vars] = A
    tableau[:n_cons, n_vars:n_vars + n_cons] = np.eye(n_cons)
    tableau[:n_cons, -1] = b
    tableau[-1, :n_vars] = -c
    basic = slack_names.copy()

    def get_point(tableau, basic):
        sol = {name: 0.0 for name in var_names}
        for i, name in enumerate(basic):
            if name in sol:
                sol[name] = tableau[i, -1]
        return sol, tableau[-1, -1]

    steps = []
    pt, z = get_point(tableau, basic)
    steps.append({"label": "จุดเริ่มต้น (origin)", "point": pt, "Z": z, "tableau": tableau.copy(), "basic": basic.copy()})

    it = 0
    while True:
        z_row = tableau[-1, :-1]
        if np.all(z_row >= -1e-9):
            break
        entering = np.argmin(z_row)
        col = tableau[:n_cons, entering]
        rhs = tableau[:n_cons, -1]
        ratios = np.where(col > 1e-9, rhs / np.where(col > 1e-9, col, 1), np.inf)
        leaving = np.argmin(ratios)
        pivot_val = tableau[leaving, entering]

        it += 1
        entering_name, leaving_name = all_names[entering], basic[leaving]

        before_pt, before_z = get_point(tableau, basic)

        tableau[leaving, :] /= pivot_val
        for r in range(len(tableau)):
            if r != leaving:
                tableau[r, :] -= tableau[r, entering] * tableau[leaving, :]
        basic[leaving] = entering_name

        after_pt, after_z = get_point(tableau, basic)

        print(f"### Pivot {it}: {entering_name} เข้าแทน {leaving_name}")
        print(f"  ก่อน pivot : point={tuple(round(float(v),1) for v in before_pt.values())}  Z={before_z:.1f}")
        print(f"  หลัง pivot : point={tuple(round(float(v),1) for v in after_pt.values())}  Z={after_z:.1f}")
        print(f"  --> Z {'เพิ่มขึ้น' if after_z > before_z else 'เท่าเดิม'} จาก {before_z:.1f} เป็น {after_z:.1f}\n")

        steps.append({"label": f"หลัง pivot {it}: {entering_name} เข้าแทน {leaving_name}",
                      "point": after_pt, "Z": after_z, "tableau": tableau.copy(), "basic": basic.copy()})

    return steps


steps = simplex_maximize(c, A, b, var_names=["x1", "x2"])
print(f"จำนวน pivot ทั้งหมด: {len(steps)-1} ครั้ง (จาก {len(corners)} จุดมุมทั้งหมดในกราฟ)")


### Pivot 1: x2 เข้าแทน S2
  ก่อน pivot : point=(0.0, 0.0)  Z=0.0
  หลัง pivot : point=(0.0, 6.0)  Z=30.0
  --> Z เพิ่มขึ้น จาก 0.0 เป็น 30.0

### Pivot 2: x1 เข้าแทน S3
  ก่อน pivot : point=(0.0, 6.0)  Z=30.0
  หลัง pivot : point=(2.0, 6.0)  Z=36.0
  --> Z เพิ่มขึ้น จาก 30.0 เป็น 36.0

จำนวน pivot ทั้งหมด: 2 ครั้ง (จาก 5 จุดมุมทั้งหมดในกราฟ)


In [17]:
# 📈 Visualization: Simplex Progress - Z value improvement บนกราฟ
step_nums = list(range(len(steps)))
z_values = [s["Z"] for s in steps]
step_labels = [s["label"] for s in steps]

fig_z = go.Figure()
fig_z.add_trace(go.Scatter(
    x=step_nums, y=z_values, mode="lines+markers",
    line={"color": "green", "width": 3},
    marker={"size": 12, "color": "green", "symbol": "circle"},
    text=step_labels,
    hovertemplate="<b>%{text}</b><br>Iteration %{x}<br>Z = %{y:.0f}<extra></extra>",
    name="Z Progression",
    fill="tozeroy", fillcolor="rgba(0,200,100,0.2)",
))

fig_z.update_layout(
    title="<b>📈 Simplex Progress: ค่า Z เพิ่มขึ้นทีละขั้น (Never Decreasing!)</b>",
    xaxis_title="Iteration (Pivot Number)", yaxis_title="Z Value",
    xaxis={"tickmode": "linear", "dtick": 1},
    width=700, height=400, hovermode="x unified", template="plotly_white",
)
fig_z.show()

# ตารางสรุปการเดินของ Simplex
simplex_path = pd.DataFrame({
    "Step": step_nums,
    "Label": step_labels,
    "Point (x₁, x₂)": [f"({s['point']['x1']:.1f}, {s['point']['x2']:.1f})" for s in steps],
    "Z Value": [f"{s['Z']:.0f}" for s in steps],
    "Z Increase": ["Start"] + [f"Δ = {steps[i]['Z'] - steps[i-1]['Z']:.0f}" for i in range(1, len(steps))],
})
print("\n=== 🚶 Simplex's Walking Path (ทีละ pivot) ===")
print(simplex_path.to_string(index=False))

ValueError: 
    Invalid value of type 'builtins.range' received for the 'x' property of scatter
        Received value: range(0, 3)

    The 'x' property is an array that may be specified as a tuple,
    list, numpy array, or pandas Series

## ส่วนที่ 3 — รวมภาพ: Simplex เดินบนกราฟเดียวกับ Graphical Method

นำเส้นทางที่ Simplex เดิน (จาก `steps`) มาซ้อนทับบนกราฟ feasible region เดียวกับ Graphical Method
เพื่อพิสูจน์ให้เห็นว่ามันคือกระบวนการเดียวกัน เพียงแค่มองจากมุมต่างกัน


In [15]:
def plot_teaching_demo(A, b, c, corners, steps):
    fig = base_figure(A, b)

    visited = {(round(s["point"]["x1"], 4), round(s["point"]["x2"], 4)) for s in steps}

    # จุดมุมที่ Simplex "ไม่ได้" เดินผ่าน (สีเทา)
    skipped = [p for p in corners if (round(p[0], 4), round(p[1], 4)) not in visited]
    if skipped:
        sxs, sys_ = zip(*skipped)
        szs = [c[0] * x + c[1] * y for x, y in skipped]
        fig.add_trace(go.Scatter(
            x=sxs, y=sys_, mode="markers+text",
            marker={"size": 11, "color": "lightgray", "line": {"width": 1.5, "color": "gray"}},
            text=[f"Z={z:.0f}" for z in szs], textposition="bottom center", textfont={"color": "gray"},
            name="จุดมุมที่ Simplex ไม่เดินผ่าน",
        ))

    # เส้นทางที่ Simplex เดินจริง (สีแดง)
    path_x = [s["point"]["x1"] for s in steps]
    path_y = [s["point"]["x2"] for s in steps]
    fig.add_trace(go.Scatter(
        x=path_x, y=path_y, mode="lines+markers+text",
        line={"color": "red", "width": 3}, marker={"size": 15, "color": "red", "symbol": "star",
                                                     "line": {"width": 2, "color": "darkred"}},
        text=[f"Step {i}<br>Z={s['Z']:.0f}" for i, s in enumerate(steps)], textposition="top center",
        textfont={"color": "darkred"},
        name="เส้นทางที่ Simplex เดินจริง",
    ))

    fig.update_layout(
        title="<b>🎯 Graphical (จุดเทา=หาทุกจุด) vs Simplex (เส้นแดง=เดินทีละก้าว)</b>",
    )
    fig.show()

    print(f"🎯 Simplex เดินผ่าน {len(visited)}/{len(corners)} จุดมุม "
          f"({len(visited)/len(corners)*100:.0f}% ของทั้งหมด) แล้วเจอ optimal พอดี")


plot_teaching_demo(A, b, c, corners, steps)

## ส่วนที่ 4 — สรุปบทเรียน

| จุดมุมทั้งหมด (Graphical) | Z | Simplex เดินผ่านไหม? |
|---|---|---|
| (0,0) | 0 | ✅ จุดเริ่มต้น |
| (4,0) | 12 | ❌ ไม่เดินผ่าน |
| (4,3) | 27 | ❌ ไม่เดินผ่าน |
| (0,6) | 30 | ✅ หลัง pivot 1 |
| (2,6) | 36 | ✅ หลัง pivot 2 (Optimal) |

**ข้อสรุปสำคัญสำหรับสอนนักเรียน:**

1. **Graphical Method** ต้องหาจุดมุม**ทุกจุด** (5 จุดในที่นี้) แล้วค่อยเทียบ — ทำงานแบบ "มองภาพรวมทั้งหมดก่อนตัดสินใจ"
2. **Simplex Method** เดินผ่านแค่ **3 จุดจาก 5 จุด** (60% ของจุดมุมทั้งหมด) — ทำงานแบบ "เดินเฉพาะเส้นทางที่ดีขึ้นเรื่อยๆ ไม่เสียเวลาไปดูจุดที่ไม่มีทางดีกว่า"
3. ค่า Z ที่ Simplex เจอ **เพิ่มขึ้นทุกครั้ง** ($0\to30\to36$) ไม่เคยลดลง — นี่คือเหตุผลที่ Simplex รับประกันว่าจะเจอ optimal โดยไม่ต้องย้อนกลับ
4. ทั้งสองวิธี**ได้คำตอบเดียวกันเป๊ะ** ($x_1=2,x_2=6,Z=36$) เพราะมันคือปัญหาเดียวกัน แค่มองจากคนละมุม

> 💡 **สำหรับผู้สอน:** ยิ่งปัญหามีตัวแปร/ข้อจำกัดมากขึ้น จำนวนจุดมุมจะเพิ่มแบบ exponential (Graphical วาดไม่ได้เกิน 2-3 ตัวแปร)
> แต่ Simplex ยังคง "เดินเฉพาะเส้นทางที่ดีขึ้น" ได้เสมอ — นี่คือเหตุผลที่ Simplex ใช้แก้ปัญหาจริงที่มีตัวแปรหลักพันหลักหมื่นได้ ในขณะที่ Graphical ทำไม่ได้เลย


In [ ]:
# 📊 Bar Chart: ทุกจุดมุมเปรียบเทียบค่า Z
visited = {(round(s["point"]["x1"], 4), round(s["point"]["x2"], 4)) for s in steps}
corner_z_values = sorted(((pt, c[0] * pt[0] + c[1] * pt[1]) for pt in corners), key=lambda x: x[1])

corner_labels = [f"({pt[0]:.0f},{pt[1]:.0f})" for pt, z in corner_z_values]
corner_z = [z for pt, z in corner_z_values]
is_visited = [(round(pt[0], 4), round(pt[1], 4)) in visited for pt, z in corner_z_values]

fig_bar = go.Figure()
fig_bar.add_trace(go.Bar(
    x=corner_labels, y=corner_z,
    marker={"color": ["red" if v else "lightgray" for v in is_visited],
            "line": {"width": 2, "color": ["darkred" if v else "gray" for v in is_visited]}},
    text=[f"Z={z:.0f}" + (" ✓" if v else "") for z, v in zip(corner_z, is_visited)],
    textposition="outside",
    hovertemplate="<b>จุดมุม: %{x}</b><br>Z = %{y:.0f}<extra></extra>",
))

fig_bar.update_layout(
    title="<b>📊 ทุกจุดมุมเปรียบเทียบค่า Z (สีแดง ✓ = Simplex เดินผ่าน, สีเทา = ข้าม)</b>",
    xaxis_title="จุดมุม (x₁, x₂)", yaxis_title="Z Value",
    width=750, height=400, template="plotly_white", showlegend=False,
)
fig_bar.show()

best_pt, best_z = corner_z_values[-1]
print(f"\n✓ Optimal point: ({best_pt[0]:.0f}, {best_pt[1]:.0f}) with Z = {best_z:.0f}")


✓ Optimal point: (np.float64(2.0), np.float64(6.0)) with Z = 36


In [ ]:
# 📊 Final Visualization: Objective Function (Level Curves)
fig_obj = base_figure(A, b)

# เส้นระดับของ Objective Function (Z = constant)
xs_line = np.linspace(*PLOT_RANGE, 200)
best_z = max(c[0] * p[0] + c[1] * p[1] for p in corners)
for i, z_val in enumerate(np.linspace(0, best_z, 8)):
    yy = (z_val - c[0] * xs_line) / c[1]
    fig_obj.add_trace(go.Scatter(
        x=xs_line, y=yy, mode="lines",
        line={"color": "lightblue", "width": 1, "dash": "dot"},
        name="เส้นระดับ Z=constant", showlegend=(i == 0),
        hovertemplate=f"Z={z_val:.0f}<extra></extra>",
    ))

# ลูกศรทิศทางที่ Z เพิ่มขึ้น (ทิศทาง gradient ของ c)
grad_norm = np.array(c) / np.linalg.norm(c) * 1.5
fig_obj.add_annotation(
    x=0.3 + grad_norm[0], y=0.3 + grad_norm[1], ax=0.3, ay=0.3,
    axref="x", ayref="y", xref="x", yref="y",
    showarrow=True, arrowhead=3, arrowsize=1.5, arrowwidth=3, arrowcolor="purple",
    text="",
)
fig_obj.add_annotation(
    x=0.3 + grad_norm[0]/2, y=0.3 + grad_norm[1]/2 + 0.3,
    text="ทิศทางที่ Z เพิ่มขึ้น", showarrow=False, font={"color": "purple", "size": 12},
)

# จุดมุมทั้งหมด — เน้นจุด optimal
for pt in corners:
    z_pt = c[0] * pt[0] + c[1] * pt[1]
    is_optimal = bool(abs(z_pt - best_z) < 1e-6)
    fig_obj.add_trace(go.Scatter(
        x=[pt[0]], y=[pt[1]], mode="markers+text",
        marker={"size": 16 if is_optimal else 10,
                "color": "gold" if is_optimal else "lightgray",
                "symbol": "star" if is_optimal else "circle",
                "line": {"width": 3 if is_optimal else 1, "color": "orange" if is_optimal else "gray"}},
        text=f"({pt[0]:.0f},{pt[1]:.0f})<br>Z={z_pt:.0f}" + (" ⭐" if is_optimal else ""),
        textposition="top center", textfont={"color": "orange" if is_optimal else "gray"},
        name="จุด Optimal" if is_optimal else "", showlegend=is_optimal,
        hovertemplate="<b>จุดมุม: (%{x:.0f}, %{y:.0f})</b><extra></extra>",
    ))

fig_obj.update_layout(
    title="<b>🎯 Level Curves: Objective Function Z=3x₁+5x₂</b><br>"
          "<sub>ลูกศรสีม่วงชี้ทิศทางที่ Z เพิ่มขึ้น — Simplex เดินตามทิศทางนี้ไปเรื่อยๆ</sub>",
)
fig_obj.show()

print("\n💡 Key Insight: Simplex 'เดินขึ้นเนิน' ไปตามทิศทางที่ Z เพิ่มขึ้น จนกว่าจะถึงจุดสูงสุดในพื้นที่ที่เป็นไปได้")

ValueError: 
    Invalid value of type 'numpy.bool' received for the 'showlegend' property of scatter
        Received value: np.True_

    The 'showlegend' property is a boolean and must be specified as:
      - A boolean value: True or False